# Product Confirmation Workflow

This notebook downloads DIST-ALERT products from S3, unzips them, and runs the confirmation workflow.

In [15]:
import pandas as pd
import shutil
import zipfile
import requests
from pathlib import Path
from tqdm import tqdm
import concurrent.futures
from dist_s1 import run_sequential_confirmation_of_dist_products_workflow
import multiprocessing

In [2]:
tmp_dir =  Path('tmp')
unconfirmed_products_dir =  Path('unconfirmed_products')
confirmed_products_dir =  Path('confirmed_products')

tmp_dir.mkdir(exist_ok=True)
unconfirmed_products_dir.mkdir(exist_ok=True)
confirmed_products_dir.mkdir(exist_ok=True)

In [3]:
# Load the test products CSV
csv_path = Path('amy_val_rest_of_informal_sites_10_2025.csv')
df = pd.read_csv(csv_path)
df.head()

,name,zip_url,browse_url,product_request_time,processing_duration,high_confidence_alert_threshold,mgrs_tile_id,post_date_buffer_days,stride_for_norm_param_estimation,n_workers_for_norm_param_estimation,...,model_source,memory_strategy,batch_size_for_norm_param_estimation,post_date,track_number,low_confidence_alert_threshold,n_workers_for_despeckling,device,max_pre_imgs_per_burst_mw,model_compilation
0,fire__29TNF,https://hyp3-tibet-jpl-test-contentbucket-hrat...,https://hyp3-tibet-jpl-test-contentbucket-hrat...,2025-10-05T17:33:16+00:00,1534.845,4.5,29TNF,1,7,4,...,transformer_optimized,high,32,2025-03-02,125,2.5,4,best,none,False
1,fire__29TNF,https://hyp3-tibet-jpl-test-contentbucket-hrat...,https://hyp3-tibet-jpl-test-contentbucket-hrat...,2025-10-05T17:33:16+00:00,1412.518,4.5,29TNF,1,7,4,...,transformer_optimized,high,32,2025-03-03,147,2.5,4,best,none,False
2,fire__29TNF,https://hyp3-tibet-jpl-test-contentbucket-hrat...,https://hyp3-tibet-jpl-test-contentbucket-hrat...,2025-10-05T17:33:16+00:00,816.520,4.5,29TNF,1,7,4,...,transformer_optimized,high,32,2025-03-08,45,2.5,4,best,none,False
3,fire__29TNF,https://hyp3-tibet-jpl-test-contentbucket-hrat...,https://hyp3-tibet-jpl-test-contentbucket-hrat...,2025-10-05T17:33:16+00:00,1287.035,4.5,29TNF,1,7,4,...,transformer_optimized,high,32,2025-03-09,52,2.5,4,best,none,False
4,fire__29TNF,https://hyp3-tibet-jpl-test-contentbucket-hrat...,https://hyp3-tibet-jpl-test-contentbucket-hrat...,2025-10-05T17:33:16+00:00,1648.691,4.5,29TNF,1,7,4,...,transformer_optimized,high,32,2025-03-14,125,2.5,4,best,none,False


# Download

In [4]:
def create_download_session(max_workers: int = 5) -> requests.Session:
    """Create a requests session with appropriate settings for downloads.

    Args:
        max_workers: Number of concurrent download threads (used to size connection pool)
    """
    session = requests.Session()
    session.headers.update({'User-Agent': 'dist-s1-calval/1.0'})

    pool_maxsize = max(max_workers * 2, 10)
    pool_maxsize = min(pool_maxsize, 50)

    adapter = requests.adapters.HTTPAdapter(
        pool_connections=10,
        pool_maxsize=pool_maxsize,
        max_retries=4, 
    )
    session.mount('http://', adapter)
    session.mount('https://', adapter)
    return session

def download_file(url, destination_path, session):
    with session.get(url, stream=True) as response:
        response.raise_for_status()
    
        with open(destination_path, 'wb') as file:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    file.write(chunk)
    
    return destination_path

In [5]:
records = df.to_dict(orient='records')

download_data = [{'url':r['zip_url'],
                  'dest_path': tmp_dir/ f'{Path(r['zip_url']).name}'} for r in records]
download_data[:3]

[{'url': 'https://hyp3-tibet-jpl-test-contentbucket-hratibh1y9pa.s3.us-west-2.amazonaws.com/603204c5-11cd-4fa4-84f3-6ec7e9f884d8/OPERA_L3_DIST-ALERT-S1_T29TNF_20250302T064249Z_20251005T203148Z_S1_30_v0.1.zip',
  'dest_path': PosixPath('tmp/OPERA_L3_DIST-ALERT-S1_T29TNF_20250302T064249Z_20251005T203148Z_S1_30_v0.1.zip')},
 {'url': 'https://hyp3-tibet-jpl-test-contentbucket-hratibh1y9pa.s3.us-west-2.amazonaws.com/dddc98f5-1881-4542-889e-49fd080ff2ba/OPERA_L3_DIST-ALERT-S1_T29TNF_20250303T182805Z_20251005T194634Z_S1_30_v0.1.zip',
  'dest_path': PosixPath('tmp/OPERA_L3_DIST-ALERT-S1_T29TNF_20250303T182805Z_20251005T194634Z_S1_30_v0.1.zip')},
 {'url': 'https://hyp3-tibet-jpl-test-contentbucket-hratibh1y9pa.s3.us-west-2.amazonaws.com/8ef8ba9d-a9ed-4837-aeb6-8604a6e9acfe/OPERA_L3_DIST-ALERT-S1_T29TNF_20250308T183616Z_20251005T185233Z_S1_30_v0.1.zip',
  'dest_path': PosixPath('tmp/OPERA_L3_DIST-ALERT-S1_T29TNF_20250308T183616Z_20251005T185233Z_S1_30_v0.1.zip')}]

In [6]:
N_WORKERS = 10

session = create_download_session(N_WORKERS)
def download_one_with_session(input_data) -> Path:
    url = input_data['url']
    dest_path = input_data['dest_path']
    dest_path.parent.mkdir(exist_ok=True, parents=True)
    return download_file(url, dest_path, session)
with concurrent.futures.ThreadPoolExecutor(max_workers=N_WORKERS) as executor:
    zip_paths = list(tqdm(executor.map(download_one_with_session, download_data[:]), total=len(download_data[:])))

100%|█████████████████| 514/514 [35:47<00:00,  4.18s/it]


# Unzip

In [8]:
def unzip_file(zip_path, extract_to):
    zip_path = Path(zip_path)
    extract_to = Path(extract_to)
    
    subdirectory_name = zip_path.stem
    
    full_extract_path = extract_to / subdirectory_name
    full_extract_path.mkdir(parents=True, exist_ok=True)
    
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(full_extract_path)
    
    return full_extract_path
    
for zip_path, record in zip(zip_paths, records):
    # mgrs_tile_id = downloaded_files[0].name.split('_')[3][1:]
    job_name = record['name']
    unconfirmed_products_dir = Path(f'unconfirmed_products/{job_name}')
    unconfirmed_products_dir.mkdir(exist_ok=True)
    unzip_file(zip_path, unconfirmed_products_dir)

In [9]:
ts_directories = list(Path('unconfirmed_products').glob('*/'))
ts_unzipped = [subdir.name for subdir in ts_directories]
ts_unzipped[:3]

['logging__33VWJ', 'mining__52NCF', 'shifting_cultivation__34MHE']

In [11]:
from utils import unzip_dist_s1_prod, wrap_run_sequential_confirmation_of_dist_products_workflow

In [16]:
with multiprocessing.Pool(processes=10) as pool:
    results = pool.imap(wrap_run_sequential_confirmation_of_dist_products_workflow, ts_unzipped[:])
    for _ in tqdm(results, total=len(ts_unzipped), desc="Confirming Products"):
        pass

Confirming 14 products: 100%|█| 14/14 [01:18<00:00,  5.
Confirming 24 products: 100%|█| 24/24 [01:47<00:00,  4.
Confirming 22 products: 100%|█| 22/22 [01:57<00:00,  5.
Confirming 25 products: 100%|█| 25/25 [02:15<00:00,  5.
Confirming 25 products: 100%|█| 25/25 [02:20<00:00,  5.
Confirming 26 products: 100%|█| 26/26 [02:27<00:00,  5.
Confirming 44 products: 100%|█| 44/44 [04:03<00:00,  5.
Confirming 38 products: 100%|█| 38/38 [03:25<00:00,  5.
Confirming 71 products: 100%|█| 71/71 [05:52<00:00,  4.
Confirming 77 products: 100%|█| 77/77 [06:34<00:00,  5.
Confirming 148 products: 100%|█| 148/148 [11:18<00:00, 
Confirming Products: 100%|█| 11/11 [11:24<00:00, 62.25s


In [10]:
# # Run the confirmation workflow
# for ts_dir in ts_directories:
#     print(ts_dir.name)
#     run_sequential_confirmation_of_dist_products_workflow(
#         ts_dir, 
#         confirmed_products_dir / ts_dir.name
#     )

In [ ]:
# cleanup_temp = True
# if cleanup_temp:
#     shutil.rmtree(tmp_dir)